## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

## Data generation

In [2]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = '0'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.4, -0.1, 0, 0.1, 0.4], rhos=[0.5, 0.6, 0.7, 0.8, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.77351285222025


## MI estimation

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [5]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500
finished: t= 0 loss= 1.0027254819869995 loss val= 0.9960393905639648 best val loss= 0.9960393905639648 best t= 0
finished: t= 63 loss= 0.49910247325897217 loss val= 0.49578535556793213 best val loss= 0.4736981987953186 best t= 38
finished: t= 126 loss= 0.4885149896144867 loss val= 0.522550106048584 best val loss= 0.4667281210422516 best t= 77
finished: t= 189 loss= 0.4767135977745056 loss val= 0.506729006767273 best val loss= 0.4667281210422516 best t= 77
finished: t= 252 loss= 0.48397162556648254 loss val= 0.49608999490737915 best val loss= 0.4600529074668884 best t= 201
finished: t= 315 loss= 0.4589585065841675 loss val= 0.48786813020706177 best val loss= 0.4600529074668884 best t= 201
finished: t= 378 loss= 0.49578818678855896 loss val= 0.5024564266204834 best val loss= 0.4600529074668884 best t= 201
finished: t= 441 loss= 0.5185101628303528 loss val= 0.4931364953517914 best val loss= 0.45641767978668213 best t= 429
finished: t= 504 loss= 0.4964945316314697 los

In [6]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.024753307923674583 loss val= 0.023184793069958687 best val loss= 0.023184793069958687 best t= 0
finished: t= 63 loss= -8.02322769165039 loss val= -5.759219169616699 best val loss= -5.964230537414551 best t= 59
finished: t= 126 loss= -6.107079982757568 loss val= -5.995597839355469 best val loss= -6.220268726348877 best t= 72
finished: t= 189 loss= -6.031702995300293 loss val= -6.113834381103516 best val loss= -6.313021183013916 best t= 145
finished: t= 252 loss= -7.171637535095215 loss val= -6.171250343322754 best val loss= -6.340392112731934 best t= 199
finished: t= 315 loss= -5.659792423248291 loss val= -5.592895030975342 best val loss= -6.357156753540039 best t= 297
finished: t= 378 loss= -5.242029666900635 loss val= -6.052935600280762 best val loss= -6.411496162414551 best t= 334
finished: t= 441 loss= -5.612757205963135 loss val= -5.9680891036987305 best val loss= -6.411496162414551 best t= 334
finished: t= 504 loss= -6.616899490356445 loss val= -6.1473207473

In [7]:
## Neural adaptive MI estimate
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 2.0216870307922363 loss val= 2.002016544342041 best val loss= 2.002016544342041 best t= 0
finished: t= 126 loss= 1.5680794715881348 loss val= 1.6228399276733398 best val loss= 1.6059975624084473 best t= 107
finished: t= 252 loss= 1.5848383903503418 loss val= 1.6292979717254639 best val loss= 1.6059975624084473 best t= 107


finished: t= 0 loss= 2.0713446140289307 loss val= 1.9935591220855713 best val loss= 1.9935591220855713 best t= 0
finished: t= 126 loss= 1.5928235054016113 loss val= 1.6180691719055176 best val loss= 1.6042596101760864 best t= 92
finished: t= 252 loss= 1.6433005332946777 loss val= 1.6171458959579468 best val loss= 1.6042596101760864 best t= 92


finished: t= 0 loss= 449.9018249511719 loss val= 451.107421875 best val loss= 451.107421875 best t= 0
finished: t= 101 loss= 80.15873718261719 loss val= 80.3466796875 best val loss= 80.34011840820312 best t= 67
finished: t= 202 loss= 80.0888442993164 los

In [4]:
## f-DIME
from estimators.fDIME import fDIME

estimator = fDIME(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.3973780870437622 loss val= 1.3972280025482178 best val loss= 1.3972280025482178 best t= 0
finished: t= 63 loss= 0.009949397295713425 loss val= 0.1532161831855774 best val loss= 0.07616010308265686 best t= 45
finished: t= 126 loss= 0.015142418444156647 loss val= 0.17329177260398865 best val loss= 0.07616010308265686 best t= 45
finished: t= 189 loss= 0.015375334769487381 loss val= 0.20486846566200256 best val loss= 0.07616010308265686 best t= 45


true MI: 10.77351285222025
est MI: 11.10472297668457
